# Extract M³ technical metadata from AutoDDG and DataAtlas descriptions

AutoDDG and DataAtlas only emit a free-text `description`. M³ spatial-temporal metadata (the iNaturalist Croissant subset, also used for PANGAEA-quant) is a structured record:

```json
{
  "description": "...original AutoDDG or DataAtlas description...",
  "spatialCoverage": {"min_lat": ..., "min_lon": ..., "max_lat": ..., "max_lon": ...},
  "temporalCoverage": {"from": "YYYY-MM-DDTHH:MM:SS", "to": "..."},
  "spatial": "point locations | 10 m | ...",
  "temporal": "daily | monthly | ..."
}
```

`spatialCoverage` / `temporalCoverage` are the fields scored in `03-TvdP-validate-metadata.ipynb`. `spatial` / `temporal` are the resolution strings shown in the M³ technical-metadata box.

This notebook uses **Qwen 3.6** (`Qwen/Qwen3.6-35B-A3B-FP8`) to map each existing description onto that schema. Values are taken **only from the description text** — the extractor does not see the CSV. If a field is not stated, it is stored as `null`.

Each output JSON copies the source `description` and adds the extracted technical fields. Files go under `outputs/technical_from_descriptions/{pangaea|inaturalist}/{run}/{id}_{corpus}.json`, next to `outputs/pangaea`. Extraction is checkpointed and skipped on resume. iNaturalist extraction covers both AutoDDG and DataAtlas.

In [ ]:
import json
import os
import re
from pathlib import Path
from typing import Optional

import pandas as pd
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, ConfigDict, Field
from tqdm.auto import tqdm

ENV_PATH = Path('/home/com3dian/Github/metadata_agent/.env')
load_dotenv(ENV_PATH)

SURF_API_KEY = os.getenv('SURF_API_KEY')
SURF_API_BASE = os.getenv('SURF_API_BASE')
if not SURF_API_KEY or not SURF_API_BASE:
    raise RuntimeError(f'SURF_API_KEY or SURF_API_BASE is missing from {ENV_PATH}')

METADATA_AGENT_ROOT = Path('/home/com3dian/Github/metadata_agent')
REPO_ROOT = Path('/home/com3dian/Github/process_data_metadata_agent')
BASELINE_OUTPUT_ROOT = METADATA_AGENT_ROOT / 'outputs'
OUTPUT_ROOT = REPO_ROOT / 'outputs' / 'technical_from_descriptions'
CHECKPOINT_PATH = OUTPUT_ROOT / 'extraction_checkpoint.csv'
M3_EXAMPLE_DIRS = {
    'pangaea': METADATA_AGENT_ROOT / 'outputs' / 'pangaea_inat_mistralai_Devstral-Small-2-24B-Instruct-2512',
    'inaturalist': METADATA_AGENT_ROOT / 'outputs' / 'inat_Qwen_Qwen3.6-35B-A3B-FP8',
}


In [ ]:
# PANGAEA and iNaturalist AutoDDG/DataAtlas runs.
CORPORA = {
    'pangaea': {
        'expected': 75,
        'filename_suffix': 'pangaea',
        'autoddg': [
            'autoddg_pangaea_default-text-large',
            'autoddg_pangaea_Qwen_Qwen3.6-35B-A3B-FP8',
            'autoddg_pangaea_Sehyo_Qwen3.5-122B-A10B-NVFP4',
            'autoddg_pangaea_mistralai_Devstral-Small-2-24B-Instruct-2512',
            'autoddg_pangaea_mistralai_Mistral-Small-3.2-24B-Instruct-2506',
        ],
        'dataatlas': [
            'dataatlas_pangaea_default-text-large',
            'dataatlas_pangaea_Qwen_Qwen3.6-35B-A3B-FP8',
            'dataatlas_pangaea_Sehyo_Qwen3.5-122B-A10B-NVFP4',
            'dataatlas_pangaea_mistralai_Devstral-Small-2-24B-Instruct-2512',
            'dataatlas_pangaea_mistralai_Mistral-Small-3.2-24B-Instruct-2506',
        ],
    },
    'inaturalist': {
        'expected': 100,
        'filename_suffix': 'inaturalist',
        'autoddg': [
            'autoddg_inat_default-text-large',
            'autoddg_inat_Qwen_Qwen3.6-35B-A3B-FP8',
            'autoddg_inat_Sehyo_Qwen3.5-122B-A10B-NVFP4',
            'autoddg_inat_mistralai_Devstral-Small-2-24B-Instruct-2512',
            'autoddg_inat_mistralai_Mistral-Small-3.2-24B-Instruct-2506',
        ],
        'dataatlas': [
            'dataatlas_inat_default-text-large',
            'dataatlas_inat_Qwen_Qwen3.6-35B-A3B-FP8',
            'dataatlas_inat_Sehyo_Qwen3.5-122B-A10B-NVFP4',
            'dataatlas_inat_mistralai_Devstral-Small-2-24B-Instruct-2512',
            'dataatlas_inat_mistralai_Mistral-Small-3.2-24B-Instruct-2506',
        ],
    },
}

RUNS = []
for corpus, config in CORPORA.items():
    for method, run_key in [('AutoDDG', 'autoddg'), ('DataAtlas', 'dataatlas')]:
        for run_folder in config[run_key]:
            RUNS.append({
                'corpus': corpus,
                'method': method,
                'run': run_folder,
                'suffix': run_key,
                'expected': config['expected'],
                'filename_suffix': config['filename_suffix'],
            })

EXTRACTOR_MODEL = 'Qwen/Qwen3.6-35B-A3B-FP8'

# Preview first, then set False to extract every available description.
DRY_RUN = False
DRY_RUN_PER_GROUP = 1
# Optional cap for a smoke test of the full loop. None = all available descriptions.
EXTRACT_LIMIT = None


## Inspect AutoDDG and DataAtlas descriptions

The extraction source is the final `description` field only. AutoDDG writes ~100-word paragraphs; DataAtlas writes ~12-word summaries. Numeric bounding boxes are rare in both, which is expected: the extractor should return `null` rather than invent coordinates. iNaturalist AutoDDG descriptions often mention countries and year ranges, still usually without numeric coordinates.

In [ ]:
def load_ok_results(folder: Path, suffix: str) -> pd.DataFrame:
    rows = []
    for path in sorted(folder.glob(f'*_{suffix}.json')):
        with path.open(encoding='utf-8') as handle:
            result = json.load(handle)
        if result.get('status') != 'ok':
            continue
        description = result.get('description')
        if not isinstance(description, str) or not description.strip():
            continue
        rows.append({
            'path': str(path),
            'id': result.get('dataset_id', path.name.split('_')[0]),
            'model': result.get('model'),
            'description': description.strip(),
            'n_words': len(description.split()),
        })
    return pd.DataFrame(rows)


def mention_flags(text: str) -> pd.Series:
    has_year = bool(re.search(r'\b(19|20)\d{2}\b', text))
    has_coord_words = bool(re.search(
        r'\b(lat|lon|latitude|longitude|coordinate|bounding box|WGS)\b',
        text,
        flags=re.I,
    ))
    has_number = bool(re.search(r'-?\d+\.\d+', text))
    return pd.Series({
        'mentions_year': has_year,
        'mentions_coord_terms': has_coord_words,
        'mentions_decimal': has_number,
    })


coverage_rows = []
preview_by_run = {}
for item in RUNS:
    predictions = load_ok_results(BASELINE_OUTPUT_ROOT / item['run'], item['suffix'])
    if not predictions.empty:
        predictions = predictions.copy()
        predictions['corpus'] = item['corpus']
        predictions['method'] = item['method']
        predictions['filename_suffix'] = item['filename_suffix']
    preview_by_run[item['run']] = predictions
    if predictions.empty:
        flags = pd.DataFrame(columns=['mentions_year', 'mentions_coord_terms', 'mentions_decimal'])
    else:
        flags = predictions['description'].apply(mention_flags)
    coverage_rows.append({
        'corpus': item['corpus'],
        'method': item['method'],
        'run': item['run'],
        'available': len(predictions),
        'expected': item['expected'],
        'median_words': float(predictions['n_words'].median()) if not predictions.empty else None,
        'mentions_year': int(flags['mentions_year'].sum()) if not flags.empty else 0,
        'mentions_coord_terms': int(flags['mentions_coord_terms'].sum()) if not flags.empty else 0,
    })

coverage = pd.DataFrame(coverage_rows)
display(coverage)


In [ ]:
# Side-by-side: PANGAEA AutoDDG/DataAtlas, iNaturalist AutoDDG, and M³ technical records.
example_ids = {}
for corpus, autoddg_run, dataatlas_run in [
    ('pangaea', 'autoddg_pangaea_Qwen_Qwen3.6-35B-A3B-FP8', 'dataatlas_pangaea_Qwen_Qwen3.6-35B-A3B-FP8'),
    ('inaturalist', 'autoddg_inat_Qwen_Qwen3.6-35B-A3B-FP8', 'dataatlas_inat_Qwen_Qwen3.6-35B-A3B-FP8'),
]:
    autoddg_df = preview_by_run.get(autoddg_run, pd.DataFrame())
    if autoddg_df.empty:
        print(f'{corpus}: no AutoDDG descriptions')
        continue
    example_id = str(autoddg_df['id'].iloc[0])
    example_ids[corpus] = example_id
    print(f'\n===== {corpus} | dataset {example_id} =====')
    print('\n--- AutoDDG description ---')
    print(autoddg_df.loc[autoddg_df['id'].astype(str) == example_id, 'description'].iloc[0])
    if dataatlas_run:
        dataatlas_df = preview_by_run.get(dataatlas_run, pd.DataFrame())
        match = dataatlas_df.loc[dataatlas_df['id'].astype(str) == example_id, 'description'] if not dataatlas_df.empty else pd.Series(dtype=str)
        print('\n--- DataAtlas description ---')
        print(match.iloc[0] if not match.empty else '(missing)')
    else:
        print('\n--- DataAtlas description ---')
        print('(no DataAtlas iNaturalist run)')

    example_dir = M3_EXAMPLE_DIRS.get(corpus)
    filename_suffix = CORPORA[corpus]['filename_suffix']
    m3_path = example_dir / f'{example_id}_{filename_suffix}.json' if example_dir else None
    if m3_path is not None and m3_path.exists():
        with m3_path.open(encoding='utf-8') as handle:
            m3_example = json.load(handle)
        print(f'\n--- M³ spatial-temporal schema example ({m3_path.name}) ---')
        print(json.dumps(m3_example, indent=2))
    elif example_dir is not None and example_dir.is_dir():
        m3_files = sorted(example_dir.glob(f'*_{filename_suffix}.json'))
        if m3_files:
            with m3_files[0].open(encoding='utf-8') as handle:
                m3_example = json.load(handle)
            print(f'\n--- M³ spatial-temporal schema example ({m3_files[0].name}) ---')
            print(json.dumps(m3_example, indent=2))


## Qwen 3.6 extractor

The schema matches the M³ Croissant technical fields: a WGS84 bounding box, an ISO 8601 interval, and optional resolution strings. The model is instructed not to geocode place names or complete missing dates from world knowledge.

In [ ]:
class SpatialCoverage(BaseModel):
    min_lat: float = Field(description='Southern bound in WGS84 decimal degrees')
    min_lon: float = Field(description='Western bound in WGS84 decimal degrees')
    max_lat: float = Field(description='Northern bound in WGS84 decimal degrees')
    max_lon: float = Field(description='Eastern bound in WGS84 decimal degrees')


class TemporalCoverage(BaseModel):
    model_config = ConfigDict(populate_by_name=True)
    from_: str = Field(
        alias='from',
        description='Start of the covered period as an ISO 8601 date or datetime',
    )
    to: str = Field(description='End of the covered period as an ISO 8601 date or datetime')


class TechnicalMetadata(BaseModel):
    spatialCoverage: Optional[SpatialCoverage] = Field(
        default=None,
        description='Geographic bounding box. Null if the description has no numeric coordinates.',
    )
    temporalCoverage: Optional[TemporalCoverage] = Field(
        default=None,
        description='Time interval with keys from and to. Null if no dates are stated.',
    )
    spatial: Optional[str] = Field(
        default=None,
        description='Spatial resolution if stated, e.g. point locations, 10 m, 0.25 degree.',
    )
    temporal: Optional[str] = Field(
        default=None,
        description='Temporal resolution if stated, e.g. daily, monthly, yearly.',
    )


EXTRACTION_SYSTEM = '''You extract Croissant technical metadata from a dataset description.

Return only the four M³ technical fields:
- spatialCoverage: {min_lat, min_lon, max_lat, max_lon} in WGS84 decimal degrees
- temporalCoverage: {from, to} as ISO 8601 date or datetime strings
- spatial: spatial resolution as a short string
- temporal: temporal resolution as a short string

Rules:
- Use only facts explicitly stated in the description. Do not use outside knowledge.
- Do not geocode place names such as Baltic Sea, Taiwan, or Senegal into coordinates.
- Populate spatialCoverage only when numeric latitudes and longitudes are present.
- Populate temporalCoverage only when start and end dates or years are present. A single year may be used for both from and to as YYYY-01-01T00:00:00 and YYYY-12-31T23:59:59.
- If a field is not supported by the description, return null for that field.
- Do not invent bounding boxes, timestamps, or resolutions.'''


llm = ChatOpenAI(
    model=EXTRACTOR_MODEL,
    api_key=SURF_API_KEY,
    base_url=SURF_API_BASE,
    default_headers={'X-API-KEY': SURF_API_KEY},
    temperature=0,
    max_tokens=1024,
    extra_body={'chat_template_kwargs': {'enable_thinking': False}},
)


In [ ]:
def _dump_technical(metadata: TechnicalMetadata) -> dict:
    payload = metadata.model_dump(by_alias=True)
    temporal = payload.get('temporalCoverage')
    if isinstance(temporal, dict) and 'from_' in temporal and 'from' not in temporal:
        temporal['from'] = temporal.pop('from_')
    return payload


def _parse_json_object(text: str) -> dict:
    stripped = text.strip()
    fenced = re.search(r'```(?:json)?\s*(\{.*?\})\s*```', stripped, flags=re.S)
    if fenced:
        stripped = fenced.group(1)
    else:
        start = stripped.find('{')
        end = stripped.rfind('}')
        if start == -1 or end == -1 or end <= start:
            raise ValueError(f'No JSON object in model response: {text[:300]}')
        stripped = stripped[start:end + 1]
    return json.loads(stripped)


def extract_technical_metadata(description: str) -> dict:
    messages = [
        ('system', EXTRACTION_SYSTEM),
        ('human', 'Description:\n' + description + '\n\nExtract the technical metadata.'),
    ]
    last_error = None
    for method in ('json_schema', 'function_calling', None):
        try:
            kwargs = {} if method is None else {'method': method}
            structured = llm.with_structured_output(TechnicalMetadata, **kwargs)
            result = structured.invoke(messages)
            if isinstance(result, TechnicalMetadata):
                return _dump_technical(result)
            if isinstance(result, dict):
                return _dump_technical(TechnicalMetadata.model_validate(result))
        except Exception as error:
            last_error = error

    response = llm.invoke(messages)
    content = response.content if hasattr(response, 'content') else str(response)
    try:
        parsed = _parse_json_object(content)
        return _dump_technical(TechnicalMetadata.model_validate(parsed))
    except Exception as error:
        raise RuntimeError(
            f'Structured extraction failed: {last_error}; JSON fallback failed: {error}'
        ) from error


def output_dir_for(corpus: str, run_folder: str) -> Path:
    return OUTPUT_ROOT / corpus / run_folder


def output_path_for(corpus: str, run_folder: str, dataset_id, filename_suffix: str) -> Path:
    return output_dir_for(corpus, run_folder) / f'{dataset_id}_{filename_suffix}.json'


def empty_technical_record() -> dict:
    return {
        'spatialCoverage': None,
        'temporalCoverage': None,
        'spatial': None,
        'temporal': None,
    }


def output_record(description: str, extracted: dict) -> dict:
    return {
        'description': description,
        **extracted,
    }


## Preview on one AutoDDG and one DataAtlas description

This cell calls Qwen 3.6 on one PANGAEA AutoDDG description, one PANGAEA DataAtlas description, one iNaturalist AutoDDG description, and one iNaturalist DataAtlas description. Leave `DRY_RUN = True` until the previews look right.

In [ ]:
preview_records = []
preview_runs = [
    'autoddg_pangaea_Qwen_Qwen3.6-35B-A3B-FP8',
    'dataatlas_pangaea_Qwen_Qwen3.6-35B-A3B-FP8',
    'autoddg_inat_Qwen_Qwen3.6-35B-A3B-FP8',
    'dataatlas_inat_Qwen_Qwen3.6-35B-A3B-FP8',
]
for run_folder in preview_runs:
    predictions = preview_by_run.get(run_folder, pd.DataFrame())
    if predictions.empty:
        print(f'{run_folder}: no descriptions')
        continue
    corpus = predictions['corpus'].iloc[0]
    example_id = example_ids.get(corpus)
    if example_id is not None and (predictions['id'].astype(str) == example_id).any():
        row = predictions.loc[predictions['id'].astype(str) == example_id].iloc[0]
    else:
        row = predictions.iloc[0]
    extracted = extract_technical_metadata(row['description'])
    record = output_record(row['description'], extracted)
    preview_records.append({
        'corpus': corpus,
        'run': run_folder,
        'id': row['id'],
        'record': record,
    })
    print(f'\n=== {corpus} | {run_folder} | dataset {row["id"]} ===')
    print(json.dumps(record, indent=2, ensure_ascii=False))

preview_records


## Extract all descriptions

With `DRY_RUN = True` this cell only extracts `DRY_RUN_PER_GROUP` records per corpus/method group. Set `DRY_RUN = False` to process every available PANGAEA and iNaturalist description. Existing JSON files are skipped.

In [ ]:
def jobs_for_runs() -> list[dict]:
    jobs = []
    for item in RUNS:
        predictions = preview_by_run[item['run']]
        for row in predictions.itertuples(index=False):
            jobs.append({
                'corpus': item['corpus'],
                'method': item['method'],
                'run': item['run'],
                'id': row.id,
                'model': row.model,
                'description': row.description,
                'source_path': row.path,
                'output_path': str(output_path_for(
                    item['corpus'],
                    item['run'],
                    row.id,
                    item['filename_suffix'],
                )),
            })
    return jobs


jobs = jobs_for_runs()

if DRY_RUN:
    limited = []
    for (corpus, method), group in pd.DataFrame(jobs).groupby(['corpus', 'method'], sort=False):
        limited.extend(group.head(DRY_RUN_PER_GROUP).to_dict('records'))
    jobs = limited
elif EXTRACT_LIMIT is not None:
    jobs = jobs[:EXTRACT_LIMIT]

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
checkpoint_rows = []
if CHECKPOINT_PATH.exists() and not DRY_RUN:
    checkpoint_rows = pd.read_csv(CHECKPOINT_PATH).to_dict('records')
    already = {(str(row['corpus']), str(row['run']), str(row['id'])) for row in checkpoint_rows}
else:
    already = set()

n_skipped = 0
n_written = 0
n_failed = 0
for job in tqdm(jobs, desc='Extract technical metadata'):
    key = (str(job['corpus']), str(job['run']), str(job['id']))
    out_path = Path(job['output_path'])
    if key in already and out_path.exists():
        n_skipped += 1
        continue

    try:
        extracted = extract_technical_metadata(job['description'])
        status = 'ok'
        error = None
    except Exception as error:
        extracted = empty_technical_record()
        status = 'failed'
        error = str(error)
        n_failed += 1

    record = output_record(job['description'], extracted)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with out_path.open('w', encoding='utf-8') as handle:
        json.dump(record, handle, indent=2, ensure_ascii=False)
        handle.write('\n')

    spatial = record.get('spatialCoverage') or {}
    temporal = record.get('temporalCoverage') or {}
    checkpoint_rows.append({
        'corpus': job['corpus'],
        'method': job['method'],
        'run': job['run'],
        'id': job['id'],
        'extractor_model': EXTRACTOR_MODEL,
        'status': status,
        'error': error,
        'has_spatialCoverage': bool(spatial),
        'has_temporalCoverage': bool(temporal),
        'has_spatial': record.get('spatial') is not None,
        'has_temporal': record.get('temporal') is not None,
        'min_lat': spatial.get('min_lat') if isinstance(spatial, dict) else None,
        'min_lon': spatial.get('min_lon') if isinstance(spatial, dict) else None,
        'max_lat': spatial.get('max_lat') if isinstance(spatial, dict) else None,
        'max_lon': spatial.get('max_lon') if isinstance(spatial, dict) else None,
        'from': temporal.get('from') if isinstance(temporal, dict) else None,
        'to': temporal.get('to') if isinstance(temporal, dict) else None,
        'spatial': record.get('spatial'),
        'temporal': record.get('temporal'),
        'output_path': str(out_path),
        'source_path': job['source_path'],
    })
    already.add(key)
    n_written += 1
    if not DRY_RUN:
        pd.DataFrame(checkpoint_rows).to_csv(CHECKPOINT_PATH, index=False)

if checkpoint_rows:
    df_checkpoint = pd.DataFrame(checkpoint_rows)
    if not DRY_RUN:
        df_checkpoint.to_csv(CHECKPOINT_PATH, index=False)
        print(f'Saved checkpoint: {CHECKPOINT_PATH}')
    print(f'Wrote {n_written} files, skipped {n_skipped}, failed {n_failed}')
    display(df_checkpoint.tail(10))
else:
    df_checkpoint = pd.DataFrame()
    print('No extraction jobs ran.')


## Extraction coverage

For each baseline run, this reports how often Qwen 3.6 could populate each technical field from the description alone. Compare this with M³, which reads the table rather than the text.

The JSON files keep the source `description` and add the M³ technical fields. After `DRY_RUN = False`, check:

- `outputs/technical_from_descriptions/pangaea/{run}/{id}_pangaea.json`
- `outputs/technical_from_descriptions/inaturalist/{run}/{id}_inaturalist.json`

In [ ]:
if df_checkpoint.empty:
    raise RuntimeError('Run the extraction cell first.')

summary = (
    df_checkpoint.groupby(['corpus', 'method', 'run'], dropna=False)
    .agg(
        n=('id', 'size'),
        failed=('status', lambda s: int((s == 'failed').sum())),
        spatialCoverage=('has_spatialCoverage', 'mean'),
        temporalCoverage=('has_temporalCoverage', 'mean'),
        spatial=('has_spatial', 'mean'),
        temporal=('has_temporal', 'mean'),
    )
    .reset_index()
)
for column in ['spatialCoverage', 'temporalCoverage', 'spatial', 'temporal']:
    summary[column] = (100 * summary[column]).round(1)

summary = summary.rename(columns={
    'spatialCoverage': 'spatialCoverage [%]',
    'temporalCoverage': 'temporalCoverage [%]',
    'spatial': 'spatial resolution [%]',
    'temporal': 'temporal resolution [%]',
})
display(summary.sort_values(['corpus', 'method', 'run']).reset_index(drop=True))

print(f'\nJSON files are under {OUTPUT_ROOT}')
print('PANGAEA: {corpus}/{run}/{dataset_id}_pangaea.json')
print('iNaturalist: {corpus}/{run}/{dataset_id}_inaturalist.json')
